In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset

In [3]:
dataset_name = "afzalmengal/nfl-4th-downs-dataset"
dataset = load_dataset(dataset_name)
# check if this is a DatasetDict or a Dataset
if isinstance(dataset, dict):
    # concatenate the train and test splits into one dataset
    dataset = pd.concat([dataset[split].to_pandas() for split in dataset.keys()])
else:
    dataset = dataset.to_pandas()

In [4]:
# Exclude rows with nan in important columns
cols = ['season', 'ydstogo', 'yardline_100', 'label']
dataset = dataset.dropna(subset=cols)

In [5]:
# Split off the years we want to plot
# Compare the first 3 years of the dataset to the middle 3 years and the last 3 years of the dataset

first_years = [1999, 2000, 2001]
middle_years = [2011, 2012, 2013]
last_years = [2024, 2025, 2026]

first_years_df = dataset[dataset['season'].isin(first_years)]
middle_years_df = dataset[dataset['season'].isin(middle_years)]
last_years_df = dataset[dataset['season'].isin(last_years)]

In [9]:
max_ydstogo = dataset['ydstogo'].max()
min_ydstogo = dataset['ydstogo'].min()
colours = ['navy', 'orange', 'maroon']
colour_map = {action: colour for action, colour in zip(dataset['label'].unique(), colours)}
fig, ax = plt.subplots(3, 1, figsize=(10, 12))
# Make the plots look like a football field, with green field, white lines with yard numbers, and the end zones with the team names. The field should be 100 yards long and 53 yards wide, with the end zones being 10 yards deep.
for i in range(3):
    ax[i].set_facecolor('green')
    ax[i].set_xlim(-10, 110)
    ax[i].set_ylim(min_ydstogo, max_ydstogo)
    # Draw the yard lines
    for yard in range(0, 101, 10):
        ax[i].axvline(x=yard, color='white', linewidth=2)
        if yard > 0 and yard < 100:
            ax[i].text(yard, max_ydstogo - (max_ydstogo - min_ydstogo) * 0.1, str(yard), color='white', fontsize=8, fontweight='bold', ha='center', va='center')
            ax[i].text(yard, min_ydstogo + (max_ydstogo - min_ydstogo) * 0.1, str(yard), color='white', fontsize=8, fontweight='bold', ha='center', va='center', rotation=180)
    # Draw the end zones (dark orange with navy blue text for the Broncos, and navy blue with neon green text for the Seahawks)
    ax[i].axvspan(-10, 0, color='darkorange')
    ax[i].axvspan(100, 110, color='navy')
    ax[i].text(-5, (max_ydstogo + min_ydstogo) / 2, 'Endzone', color='navy', fontsize=20, fontweight='bold', ha='center', va='center', rotation=90)
    ax[i].text(105, (max_ydstogo + min_ydstogo) / 2, 'Endzone', color='darkorange', fontsize=20, fontweight='bold', ha='center', va='center', rotation=270)
    # Label the axes
    ax[i].set_xlabel('Yardline (100 = Opponent End Zone)', fontsize=12)
    ax[i].set_ylabel('Yards to Go', fontsize=12)
# Plot the actual actions
ax[0].scatter(first_years_df['yardline_100'], first_years_df['ydstogo'], c=first_years_df['label'].map(colour_map), label='First 3 Years', alpha=0.5, s=10)
ax[0].set_title('1999-2001', fontsize=14, fontweight='bold')
ax[1].scatter(middle_years_df['yardline_100'], middle_years_df['ydstogo'], c=middle_years_df['label'].map(colour_map), label='Middle 3 Years', alpha=0.5, s=10)
ax[1].set_title('2011-2013', fontsize=14, fontweight='bold')
ax[2].scatter(last_years_df['yardline_100'], last_years_df['ydstogo'], c=last_years_df['label'].map(colour_map), label='Last 3 Years', alpha=0.5, s=10)
ax[2].set_title('2024-2026', fontsize=14, fontweight='bold')
# Shared legend for both plots, outside the plots
fig.legend(handles=[plt.Line2D([0], [0], marker='o', color='w', label=action, markersize=5, markerfacecolor=colour_map[action]) for action in dataset['label'].unique()], loc='upper center', ncol=len(dataset['label'].unique()), fontsize=12)
fig.tight_layout()
plt.savefig('plot_field_year_comparison.png')
plt.close()